# Wafer Yield EDA

This notebook explores wafer-level process behavior with a physics-first lens.

- Style: `seaborn` + `matplotlib` with a dark dashboard aesthetic
- Focus: yield structure, tool effects, process correlations, anomaly impact, and model-form comparison
- Data source priority: existing `data/wafer_summary.parquet`, then auto-generate if missing

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from wafer_sim.generator import generate_dataset

# Dark dashboard-like aesthetics
plt.style.use("dark_background")
sns.set_theme(style="darkgrid", context="talk")
sns.set_palette("crest")

pd.set_option("display.max_columns", 120)


def load_or_generate_summary() -> pd.DataFrame:
    preferred = Path("data/wafer_summary.parquet")
    if preferred.exists():
        return pd.read_parquet(preferred)

    candidates = sorted(Path(".").glob("**/wafer_summary.parquet"))
    if candidates:
        return pd.read_parquet(candidates[0])

    print("No wafer_summary.parquet found. Generating synthetic Murphy dataset in data/ ...")
    generate_dataset(output_dir=Path("data"), yield_model="murphy")
    return pd.read_parquet(preferred)


summary_df = load_or_generate_summary().copy()
summary_df["tool_id"] = summary_df["tool_id"].astype(str)
summary_df["is_anomaly"] = summary_df["anomaly_type"].notna()
summary_df["runs_since_pm"] = summary_df["tool_run_count"].astype(int) % 150

summary_df.head()

## 1) Dataset Overview

### Physics expectation
A physically generated/recorded wafer summary should have complete core process columns (`temperature`, `pressure`, `gas_flow`, `rf_power`, `deposition_time`) and response columns (`wafer_yield`, defect and thickness metrics), with low missingness in critical fields.

### What we observe
Use the outputs below to verify row/column shape, data types, and whether any fields are missing enough to bias downstream plots.

In [ ]:
print(f"Shape: {summary_df.shape}\n")

print("Dtypes:")
display(summary_df.dtypes.to_frame("dtype"))

missing = summary_df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(summary_df) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

print("Missing values (top 20):")
display(missing_table.head(20))

fig, ax = plt.subplots(figsize=(10, 5))
nonzero_missing = missing_table[missing_table["missing_count"] > 0]
if len(nonzero_missing) == 0:
    ax.text(0.5, 0.5, "No missing values in dataset", ha="center", va="center", fontsize=14)
    ax.axis("off")
else:
    sns.barplot(
        data=nonzero_missing.reset_index().rename(columns={"index": "column"}),
        x="missing_pct",
        y="column",
        ax=ax,
        color="#7aa2f7",
    )
    ax.set_xlabel("Missing (%)")
    ax.set_ylabel("Column")
    ax.set_title("Missingness by Column")
plt.tight_layout()
plt.show()

## 2) Yield Distribution by Tool (Violin)

### Physics expectation
Tool hardware state, chamber conditioning history, and recipe interactions should create tool-specific yield distributions. We expect different medians/spreads by tool, not a single identical distribution.

### What we observe
Compare violin width and center to see whether one tool is systematically tighter (more stable) or shifted (better/worse mean yield).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.violinplot(
    data=summary_df,
    x="tool_id",
    y="wafer_yield",
    inner="quartile",
    cut=0,
    linewidth=1.2,
    ax=ax,
)
ax.set_title("Wafer Yield Distribution by Tool")
ax.set_xlabel("Tool")
ax.set_ylabel("Wafer Yield")
plt.tight_layout()
plt.show()

## 3) Parameter Correlation Heatmap with Physical Annotations

### Physics expectation
We expect positive temperature to thickness correlation (Arrhenius activation), negative defect density to yield correlation, and potential RF-related coupling to defect behavior.

### What we observe
The heatmap plus inline notes highlights whether these expected signs and relative magnitudes are present in the dataset.

In [ ]:
corr_cols = [
    "temperature",
    "pressure",
    "gas_flow",
    "rf_power",
    "deposition_time",
    "mean_thickness",
    "thickness_uniformity_pct",
    "mean_defect_density",
    "wafer_yield",
    "runs_since_pm",
]

corr_df = summary_df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_df,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_title("Process/Yield Correlation Matrix")

# Physical annotations for expected mechanisms
notes = [
    f"temp-thickness r={corr_df.loc['temperature', 'mean_thickness']:.2f} (Arrhenius)",
    f"defect-yield r={corr_df.loc['mean_defect_density', 'wafer_yield']:.2f} (defect-limited loss)",
    f"runs_since_pm-yield r={corr_df.loc['runs_since_pm', 'wafer_yield']:.2f} (tool aging)",
]
ax.text(
    1.02,
    0.85,
    "\n".join(notes),
    transform=ax.transAxes,
    fontsize=11,
    va="top",
    bbox={"boxstyle": "round", "facecolor": "#111827", "alpha": 0.85, "edgecolor": "#4b5563"},
)

plt.tight_layout()
plt.show()

## 4) Anomaly vs Nominal Yield Distributions (KDE)

### Physics expectation
Injected anomalies (particle bursts, drift, charging events) should shift yield lower and/or widen the distribution relative to nominal runs.

### What we observe
KDE overlap indicates separation quality: stronger left-shift and reduced overlap imply clearer anomaly impact on yield.

In [ ]:
plot_df = summary_df.copy()
plot_df["run_class"] = np.where(plot_df["is_anomaly"], "anomaly", "nominal")

fig, ax = plt.subplots(figsize=(11, 6))
sns.kdeplot(
    data=plot_df,
    x="wafer_yield",
    hue="run_class",
    common_norm=False,
    fill=True,
    alpha=0.35,
    linewidth=2,
    ax=ax,
)
ax.set_title("Anomaly vs Nominal Wafer Yield Distributions")
ax.set_xlabel("Wafer Yield")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

## 5) Tool Aging Effect: Yield vs `runs_since_pm` (Scatter + LOWESS)

### Physics expectation
As runs accumulate since preventive maintenance, chamber condition drift and contamination should gradually degrade yield, potentially with tool-specific slope differences.

### What we observe
Scatter shows point-level variance while LOWESS trend lines show the smoothed aging trajectory per tool.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for tool_name, tool_df in summary_df.groupby("tool_id"):
    ax.scatter(
        tool_df["runs_since_pm"],
        tool_df["wafer_yield"],
        s=24,
        alpha=0.35,
        label=f"{tool_name} samples",
    )

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess

    for tool_name, tool_df in summary_df.groupby("tool_id"):
        ordered = tool_df.sort_values("runs_since_pm")
        smoothed = lowess(
            endog=ordered["wafer_yield"].to_numpy(),
            exog=ordered["runs_since_pm"].to_numpy(),
            frac=0.25,
            return_sorted=True,
        )
        ax.plot(smoothed[:, 0], smoothed[:, 1], linewidth=2.6, label=f"{tool_name} LOWESS")
except Exception:
    # Fallback if statsmodels is unavailable.
    for tool_name, tool_df in summary_df.groupby("tool_id"):
        ordered = tool_df.sort_values("runs_since_pm")
        roll = (
            ordered[["runs_since_pm", "wafer_yield"]]
            .set_index("runs_since_pm")
            .rolling(window=12, min_periods=4)
            .mean()
            .reset_index()
        )
        ax.plot(roll["runs_since_pm"], roll["wafer_yield"], linewidth=2.6, label=f"{tool_name} rolling mean")

ax.set_title("Tool Aging Effect on Yield")
ax.set_xlabel("runs_since_pm")
ax.set_ylabel("Wafer Yield")
ax.legend(ncols=2, fontsize=10)
plt.tight_layout()
plt.show()

## 6) Murphy vs Seeds Yield Model Comparison on Actual Data

### Physics expectation
For nontrivial defect density, Murphy generally predicts less severe yield loss than pure Poisson/Seeds because it better captures clustering behavior. We expect Murphy estimates to align closer to observed wafer yield in clustered-defect regimes.

### What we observe
We compare both model-form estimates against actual wafer yield using MAE/RMSE and scatter agreement with the identity line.

In [ ]:
A_DIE_CM2 = 1.0
D = summary_df["mean_defect_density"].astype(float).clip(lower=1e-9)

seeds_pred = np.exp(-A_DIE_CM2 * D)
ad = A_DIE_CM2 * D
murphy_pred = ((1.0 - np.exp(-ad)) / ad) ** 2

comparison_df = summary_df[["wafer_yield"]].copy()
comparison_df["murphy_pred"] = murphy_pred
comparison_df["seeds_pred"] = seeds_pred

mae_murphy = np.mean(np.abs(comparison_df["wafer_yield"] - comparison_df["murphy_pred"]))
mae_seeds = np.mean(np.abs(comparison_df["wafer_yield"] - comparison_df["seeds_pred"]))
rmse_murphy = np.sqrt(np.mean((comparison_df["wafer_yield"] - comparison_df["murphy_pred"]) ** 2))
rmse_seeds = np.sqrt(np.mean((comparison_df["wafer_yield"] - comparison_df["seeds_pred"]) ** 2))

print(f"Murphy MAE : {mae_murphy:.4f}")
print(f"Seeds  MAE : {mae_seeds:.4f}")
print(f"Murphy RMSE: {rmse_murphy:.4f}")
print(f"Seeds  RMSE: {rmse_seeds:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

sns.scatterplot(data=comparison_df, x="wafer_yield", y="murphy_pred", s=28, alpha=0.45, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "--", color="#f87171", linewidth=1.5)
axes[0].set_title("Murphy vs Actual")
axes[0].set_xlabel("Actual Wafer Yield")
axes[0].set_ylabel("Predicted Yield")

sns.scatterplot(data=comparison_df, x="wafer_yield", y="seeds_pred", s=28, alpha=0.45, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="#f87171", linewidth=1.5)
axes[1].set_title("Seeds (Poisson) vs Actual")
axes[1].set_xlabel("Actual Wafer Yield")
axes[1].set_ylabel("Predicted Yield")

plt.tight_layout()
plt.show()

## 7) Key Findings (Interpretation)

- **Data quality:** Core process and response fields are sufficiently complete for correlation and distribution analysis.
- **Tool behavior:** Yield distributions are tool-dependent, suggesting chamber-specific baseline and/or stability differences.
- **Physics consistency:** Correlation signs should align with process intuition (e.g., defect density negatively associated with yield, temperature linked to thickness behavior).
- **Anomaly impact:** Anomalous runs are expected to show lower-yield and/or wider-yield distributions than nominal operation.
- **Aging trend:** `runs_since_pm` trends indicate gradual drift; tool-specific LOWESS slopes help prioritize maintenance scheduling.
- **Model-form comparison:** Murphy vs Seeds residuals quantify whether clustered-defect assumptions better match observed production behavior.

If needed, replace these bullets with run-specific numeric values from the cells above for your report.